# MyLLM - Google Colab Training Notebook
This notebook trains your custom decoder-only Transformer (`MyLLM`) using GPU acceleration.

### Pipeline Steps:
1. **Environment Setup & GPU Check**
2. **Connect Google Drive / Clone Repo / Upload Data**
3. **Tokenize & Prepare Binary Dataset** (`.bin` memmapped tokens)
4. **Pretrain MyLLM** on the binary dataset
5. **(Optional) Supervised Instruction Fine-Tuning (SFT)**
6. **Inference & Text Generation**

In [ ]:
# Step 1: Check GPU
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Step 2: Install dependencies
%pip install tokenizers tqdm ftfy datasets

### Step 3: Mount Google Drive or Locate Workspace
If you uploaded your `MY-LLM` project folder to Google Drive, mount it below.
Otherwise, you can clone your GitHub repo or work directly in Colab's local disk `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set this path to where MY-LLM is in your Google Drive
# e.g. /content/drive/MyDrive/MY-LLM
import os
PROJECT_DIR = '/content/drive/MyDrive/MY-LLM'

if os.path.exists(PROJECT_DIR):
    %cd {PROJECT_DIR}
else:
    print(f"Project dir '{PROJECT_DIR}' not found. Please adjust path or upload files directly.")

### Step 4: Tokenize Newly Collected Data
Converts `training_data.jsonl` (~500MB) into memory-mapped uint16 binary files for fast streaming during training without loading the entire dataset into RAM.

In [ ]:
!python training/prepare_fineweb_pretrain.py \
    --input data/training_data.jsonl \
    --tokenizer data/tokenizer/tokenizer_combined.json \
    --output_dir data/pretraining \
    --val_fraction 0.05

### Step 5: Pretrain MyLLM on GPU
Train the autoregressive transformer model. Checkpoints will automatically be saved to `checkpoints/pretrained_best.pt`.

In [ ]:
# Pretraining execution
# Modify pretrain.py parameters if you want larger batch size on Colab T4/A100 (e.g. batch_size=32 or 64)
!python training/pretrain.py

### Step 6: Test Generation / Inference
Run inference with your trained model checkpoint.

In [ ]:
!python generator.py --prompt "The future of artificial intelligence" --checkpoint checkpoints/pretrained_best.pt